# Week 5 — Personal Knowledge Worker (RAG)

1. **Assemble your files** in one place: put `.txt`, `.md`, `.docx` (Word), and optionally `.pdf` files in the `knowledge_base` folder.
2. **Vectorize in Chroma**: load documents, chunk, embed with sentence-transformers, store in Chroma.
3. **Conversational AI**: ask questions; answers are grounded in your documents (RAG). Chat UI with Gradio.

**Included:** Word documents (`.docx`) are loaded via `Docx2txtLoader` (requires `docx2txt`). Optional: connect to email inbox or other sources.


In [4]:
# Install langchain-community if missing (fixes IDE "could not be resolved" and runtime import errors)
try:
    from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, Docx2txtLoader
    from langchain_community.embeddings import HuggingFaceEmbeddings
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "langchain-community", "docx2txt"])
    from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader, Docx2txtLoader
    from langchain_community.embeddings import HuggingFaceEmbeddings

import os
import glob
from dotenv import load_dotenv
import gradio as gr
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# Path to your knowledge base (folder in the same directory as this notebook)
KB_DIR = "knowledge_base"
os.makedirs(KB_DIR, exist_ok=True)
# Sample file so the first run works; add your own .txt / .md / .pdf here
sample_path = os.path.join(KB_DIR, "readme.txt")
if not os.path.isfile(sample_path):
    with open(sample_path, "w", encoding="utf-8") as f:
        f.write("Personal Knowledge Base. Add .txt or .md files here. You can ask: What is in my knowledge base? What should I add?\n")
print(f"Knowledge base folder: {os.path.abspath(KB_DIR)}")

Knowledge base folder: c:\Users\Public\ai-engineering\llm_engineering\week5\community-contributions\kman\knowledge_base


In [5]:
# Load all documents from knowledge_base (.txt, .md, .docx, and .pdf)
documents = []
loader_kwargs = {"encoding": "utf-8"}
for pattern, loader_cls in [("**/*.txt", TextLoader), ("**/*.md", TextLoader)]:
    try:
        loader = DirectoryLoader(KB_DIR, glob=pattern, loader_cls=loader_cls, loader_kwargs=loader_kwargs)
        documents.extend(loader.load())
    except Exception as e:
        print(f"Loader {pattern}: {e}")
# Word documents (.docx) — requires: pip install docx2txt
try:
    loader_docx = DirectoryLoader(KB_DIR, glob="**/*.docx", loader_cls=Docx2txtLoader)
    documents.extend(loader_docx.load())
except Exception as e:
    print(f"Word loader (install docx2txt if needed): {e}")
try:
    loader_pdf = DirectoryLoader(KB_DIR, glob="**/*.pdf", loader_cls=PyPDFLoader)
    documents.extend(loader_pdf.load())
except Exception as e:
    print(f"PDF loader (optional): {e}")

# Chunk documents for retrieval
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = text_splitter.split_documents(documents) if documents else []
print(f"Loaded {len(documents)} document(s), {len(chunks)} chunks.")

Loaded 2 document(s), 3 chunks.


In [6]:
# Vectorize and store in Chroma (embeddings: sentence-transformers, no API key needed)
DB_DIR = "chroma_kb"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if chunks:
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_DIR)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    print(f"Chroma vector store created: {vectorstore._collection.count()} vectors in {DB_DIR}")
else:
    vectorstore = Chroma(embedding_function=embeddings, persist_directory=DB_DIR)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    print("No chunks to add; using existing or empty store.")

C:\Users\kwx1494728\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kwx1494728\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5959.22it

Chroma vector store created: 3 vectors in chroma_kb


In [7]:
# RAG chain: retrieve context from Chroma, then answer with OpenRouter LLM (OPENROUTER_API_KEY in .env)
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="google/gemini-2.0-flash-001",
    temperature=0.3,
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the following context. If the context does not contain the answer, say so briefly."),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [9]:
# Conversational AI: Gradio chat over your knowledge base
def chat(message, history):
    if not message or not message.strip():
        return "Ask a question about your documents."
    try:
        answer = rag_chain.invoke(message.strip())
        return answer
    except Exception as e:
        return f"Error: {e}"

gr.ChatInterface(chat, title="Knowledge base Q&A").launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
